In [1]:
import numpy as np
import pandas as pd
import re
import geopandas as gpd
from haversine import haversine, Unit

# 1. 바나프레소 정보만 가져오기

1. 서울 카페 정보 중 **바나프레소** 인 것만 필터링.
2. apv_date가 3년 전 이상인 것만 필터링 why? 3년 이상 영업 가게를 성공 가게라고 판단했기 때문.

In [2]:
# 카페 데이터 가져오기
cafe = pd.read_csv("C:/vs_code_prj/brewmap/data/mj/cafe.csv")
cafe.head()

,nm,type,apv_date,addr,floor,sitearea,lat,lng,flpop_type,trdar_cd,adstrd_cd
0,베레베레,커피숍,2002-09-19,서울 광진구 능동로13길 19,NaN,NaN,37.542885,127.070832,TR,3120053.0,11215710
1,덕수궁전통차전문,커피숍,2003-01-16,서울 중구 서소문로 109,2.0,46.28,37.563062,126.973280,TR,3120020.0,11140520
2,엘빠소,커피숍,2003-05-31,서울 종로구 성균관로 18,NaN,NaN,37.584406,126.997641,TR,3110021.0,11110650
3,씨애틀즈베스트커피대사관점,커피숍,2003-10-31,서울 종로구 세종대로 188,1.0,NaN,37.573190,126.977831,AD,NaN,11110615
4,본솔카페,커피숍,2004-01-17,서울 용산구 청파로47길 71,1.0,13.80,37.544840,126.966455,TR,3110069.0,11170555


In [3]:
# 바나프레소만 추출
bana_df = cafe[cafe['nm'].str.contains("바나프레소", na=False)].copy()
print(len(bana_df))

# 날짜 변환
bana_df['apv_date'] = pd.to_datetime(bana_df['apv_date'], errors='coerce')

# 오늘 기준 3년 이상(= 3년 전 이하) 된 매장만
cutoff = pd.Timestamp.today().normalize() - pd.DateOffset(years=3)
bana_df["success"] = (bana_df["apv_date"] <= cutoff).astype(int)

# 확인
bana_df["success"].value_counts()

58


success
0    48
1    10
Name: count, dtype: int64

# 2. 수요 점수 변수
2024년 4분기부터 2025년 3분기 까지의 데이터 사용 
<br>
- 2-1. 유동 인구_점심: tmzon_11_14_flpop_co
- 2-2. 유동 인구_저녁: tmzon_17_21_flpop_co
- 2-3. 유동 인구_기타 시간 평균: tmzon_00_06_flpop_co, tmzon_06_11_flpop_co, tmzon_14_17_flpop_co, tmzon_21_24_flpop_co

In [4]:
print(bana_df["flpop_type"].unique())

<StringArray>
['TR', 'AD', 'AL']
Length: 3, dtype: str


- 성공한 카페는 flpop_t.csv 만으로도 유동인구 값 가져올 수 있음

In [71]:
# 유동인구 데이터 불러오기
flpop_t = pd.read_csv("C:/vs_code_prj/brewmap/data/mj/flpop_t.csv")
print(flpop_t.columns)

Index(['stdr_yyqu_cd', 'trdar_se_cd_nm', 'trdar_cd', 'trdar_cd_nm',
       'tot_flpop_co', 'ml_flpop_co', 'fml_flpop_co', 'agrde_10_flpop_co',
       'agrde_20_flpop_co', 'agrde_30_flpop_co', 'agrde_40_flpop_co',
       'agrde_50_flpop_co', 'agrde_60_above_flpop_co', 'tmzon_00_06_flpop_co',
       'tmzon_06_11_flpop_co', 'tmzon_11_14_flpop_co', 'tmzon_14_17_flpop_co',
       'tmzon_17_21_flpop_co', 'tmzon_21_24_flpop_co', 'mon_flpop_co',
       'tues_flpop_co', 'wed_flpop_co', 'thur_flpop_co', 'fri_flpop_co',
       'sat_flpop_co', 'sun_flpop_co'],
      dtype='str')


In [72]:
# 최근 4개 분기 추출
recent_quarters = sorted(flpop_t["stdr_yyqu_cd"].unique())[-4:]

# 필터링
flpop_t = flpop_t[flpop_t["stdr_yyqu_cd"].isin(recent_quarters)]

In [73]:
# 중복(이상치) 확인
print(flpop_t[["stdr_yyqu_cd","trdar_cd"]].duplicated().sum())

0


In [77]:
# 분기 평균 집계
agg_cols = ["tmzon_11_14_flpop_co", "tmzon_17_21_flpop_co",
            "tmzon_00_06_flpop_co", "tmzon_06_11_flpop_co",
            "tmzon_14_17_flpop_co", "tmzon_21_24_flpop_co"]

flpop_t_agg = flpop_t.groupby("trdar_cd")[agg_cols].mean().reset_index()

# etc 평균 계산
etc_cols = ["tmzon_00_06_flpop_co", "tmzon_06_11_flpop_co", "tmzon_14_17_flpop_co", "tmzon_21_24_flpop_co"]
flpop_t_agg["flpop_etc"] = flpop_t_agg[etc_cols].mean(axis=1)
flpop_t_agg

,trdar_cd,tmzon_11_14_flpop_co,tmzon_17_21_flpop_co,tmzon_00_06_flpop_co,tmzon_06_11_flpop_co,tmzon_14_17_flpop_co,tmzon_21_24_flpop_co,flpop_etc
0,3001491,281088.25,402597.75,476771.75,357248.25,301798.00,273183.50,3.522504e+05
1,3001492,1766225.75,1358691.50,390051.50,1505678.50,1747818.50,357227.50,1.000194e+06
2,3001493,536464.00,654693.50,592612.50,670068.50,541661.00,403672.75,5.520037e+05
3,3001494,1734407.75,1660630.75,911272.25,1655835.25,1760761.75,671573.25,1.249861e+06
4,3001495,615172.50,891170.00,717608.50,738374.00,651910.50,483972.00,6.479662e+05
...,...,...,...,...,...,...,...,...
1645,3130323,32996.25,49594.00,84503.75,60058.00,33634.00,38989.75,5.429638e+04
1646,3130324,54661.50,76998.50,159406.00,107335.75,51321.00,73059.75,9.778062e+04
1647,3130325,41724.00,64311.50,61783.00,60552.50,42442.75,39066.75,5.096125e+04
1648,3130326,11467.50,17532.00,21130.50,21020.75,11652.00,11918.25,1.643038e+04


In [90]:
# TR 매핑
bana_old_df = bana_old_df.merge(
    flpop_t_agg[["trdar_cd", "tmzon_11_14_flpop_co", "tmzon_17_21_flpop_co", "flpop_etc"]],
    on="trdar_cd", how="left"
).rename(columns={
    "tmzon_11_14_flpop_co": "flpop_lunch",
    "tmzon_17_21_flpop_co": "flpop_dinner"
})

bana_old_df

,nm,type,apv_date,addr,floor,sitearea,lat,lng,flpop_type,trdar_cd,adstrd_cd,detail,flpop_lunch,flpop_dinner,flpop_etc
0,바나프레소 테헤란로점,커피숍,2019-05-23,서울 강남구 테헤란로 208,1.0,98.40,37.501130,127.039099,TR,3120197.0,11680650,banapresso,985131.50,882135.00,752222.8125
1,바나프레소 신논현역점,커피숍,2019-08-06,서울 서초구 사평대로 364,1.0,107.24,37.503733,127.023196,TR,3120187.0,11650531,banapresso,589152.00,788268.00,560948.1250
2,바나프레소 선릉공원점,커피숍,2021-07-23,서울 강남구 테헤란로63길 20,1.0,97.28,37.506522,127.050398,TR,3120210.0,11680590,banapresso,951358.25,983513.00,833761.1250
3,바나프레소 충무로점,커피숍,2022-09-19,서울 중구 필동로 15,1.0,148.43,37.560339,126.996257,TR,3120032.0,11140570,banapresso,357098.25,362848.50,287171.8750
4,바나프레소 홍대입구역사거리점,커피숍,2022-11-07,서울 마포구 양화로 129,1.0,33.00,37.555061,126.920694,TR,3120102.0,11440660,banapresso,409679.75,552486.25,389756.6250
5,바나프레소 양재이안점,커피숍,2023-01-31,서울 서초구 강남대로34길 7,1.0,48.33,37.483032,127.036465,TR,3120179.0,11650651,banapresso,440356.25,473092.50,400709.2500
6,바나프레소 여의도파크원점,커피숍,2021-03-18,서울 영등포구 여의대로 108,-1.0,59.16,37.525191,126.929113,TR,3120149.0,11560540,banapresso,364928.50,317197.25,246954.3125
7,바나프레소 양평역점,커피숍,2015-08-05,서울 영등포구 양산로 43,1.0,59.40,37.525761,126.887942,TR,3120135.0,11560610,banapresso,82196.00,68874.00,71096.8750
8,바나프레소 명동중앙우체국점,커피숍,2021-10-29,서울 중구 명동8나길 51,1.0,193.85,37.561080,126.982077,TR,3120028.0,11140550,banapresso,449272.25,410975.75,273562.8125
9,바나프레소 마포역점,커피숍,2015-11-17,서울 마포구 마포대로 34,1.0,33.00,37.539260,126.946152,TR,3120106.0,11440585,banapresso,117808.75,141110.00,142316.5000


In [80]:
len(bana_old_df)

10

    2-4. 학교 수
    2-5. 지식산업센터 수
    2-6. 병원 수 
    2-7. 학원 수
    2-8. 체육도장업 수 

In [28]:
academy = pd.read_csv("C:/vs_code_prj/brewmap/data/sa/seoul_data/academy_df.csv")
gym = pd.read_csv("C:/vs_code_prj/brewmap/data/sa/seoul_data/gym_df.csv")
hospital = pd.read_csv("C:/vs_code_prj/brewmap/data/sa/seoul_data/hospital_df.csv")
physical = pd.read_csv("C:/vs_code_prj/brewmap/data/sa/seoul_data/physical_df.csv")
school = pd.read_csv("C:/vs_code_prj/brewmap/data/sa/seoul_data/school.csv")
center = pd.read_csv("C:/vs_code_prj/brewmap/data/mj/industry_center.csv")
print(academy.columns)
print(gym.columns)
print(hospital.columns)
print(physical.columns)
print(school.columns)
print(center.columns)

Index(['idx', 'nm', 'addr', 'lat', 'lng'], dtype='str')
Index(['idx', 'dtlstatenm', 'addr', 'nm', 'type', 'lat', 'lng'], dtype='str')
Index(['idx', 'dtlstatenm', 'nm', 'type', 'addr', 'lat', 'lng'], dtype='str')
Index(['idx', 'dtlstatenm', 'addr', 'type', 'nm', 'lat', 'lng'], dtype='str')
Index(['idx', 'nm', 'addr', 'type', 'lat', 'lng'], dtype='str')
Index(['nm', 'type', 'addr', 'lat', 'lng'], dtype='str')


In [30]:
# academy 에 type 칼럼이 없어서 추가
academy['type'] = '학원'
academy.head()

,idx,nm,addr,lat,lng,type
0,1,클릭전원미술학원,서울특별시 마포구 와우산로 138 (창전동),37.553572,126.928993,학원
1,2,고려직업전문학교,서울특별시 동작구 노량진로 186 (노량진동),37.512575,126.946031,학원
2,3,노량진행정고시학원,서울특별시 동작구 노량진로 171 (노량진동),37.514034,126.945198,학원
3,4,박문각임용고시학원,서울특별시 동작구 노량진로 171 (노량진동),37.514034,126.945198,학원
4,5,노량진메가스터디입시어학원,서울특별시 동작구 장승배기로 171 (노량진동),37.513017,126.939945,학원


In [31]:
# 하나의 데이터프레임으로 만들기
dfs = [academy, gym, hospital, physical, school, center]

need_cols = ["type", "nm", "addr", "lat", "lng"]

# 각 df에서 필요한 컬럼만 추출(없는 컬럼은 NaN으로 생성)
picked = [d.reindex(columns=need_cols) for d in dfs]

# 세로로 합치기
buildings_df = pd.concat(picked, ignore_index=True)

buildings_df.head()

,type,nm,addr,lat,lng
0,학원,클릭전원미술학원,서울특별시 마포구 와우산로 138 (창전동),37.553572,126.928993
1,학원,고려직업전문학교,서울특별시 동작구 노량진로 186 (노량진동),37.512575,126.946031
2,학원,노량진행정고시학원,서울특별시 동작구 노량진로 171 (노량진동),37.514034,126.945198
3,학원,박문각임용고시학원,서울특별시 동작구 노량진로 171 (노량진동),37.514034,126.945198
4,학원,노량진메가스터디입시어학원,서울특별시 동작구 장승배기로 171 (노량진동),37.513017,126.939945


In [36]:
buildings_df['type'].unique()

<StringArray>
['학원', '병원', '학교', '지식산업센터']
Length: 4, dtype: str

In [46]:
test.columns

Index(['nm', 'type', 'apv_date', 'addr', 'floor', 'sitearea', 'lat', 'lng',
       'flpop_type', 'trdar_cd', 'adstrd_cd', 'flpop_lunch', 'flpop_dinner',
       'flpop_etc', '병원', '지식산업센터', '학원'],
      dtype='str')

In [91]:
# 반경 300m 안 개수 찾기

# 0) type 매핑
type_map = {
    "학원": "cnt_academy",
    "병원": "cnt_hospital",
    "학교": "cnt_school",
    "지식산업센터": "cnt_company"
}

# 1) 함수 정의
def count_buildings_nearby(cafe_lat, cafe_lon, buildings_df, radius=300):
    mask = buildings_df.apply(
        lambda row: haversine(
            (cafe_lat, cafe_lon), (row["lat"], row["lng"]), unit=Unit.METERS
        ) <= radius, axis=1
    )
    nearby = buildings_df[mask]
    counts = nearby.groupby("type").size()
    
    # 없는 type도 0으로 채우기
    counts = counts.reindex(type_map.keys(), fill_value=0)
    
    # 영어로 rename
    counts.index = [type_map[t] for t in counts.index]
    return counts

# 적용
result = bana_old_df.apply(
    lambda row: count_buildings_nearby(row["lat"], row["lng"], buildings_df),
    axis=1
).fillna(0).astype(int)

bana_old_df = pd.concat([bana_old_df, result], axis=1)
bana_old_df.head()

,nm,type,apv_date,addr,floor,sitearea,lat,lng,flpop_type,trdar_cd,adstrd_cd,detail,flpop_lunch,flpop_dinner,flpop_etc,cnt_academy,cnt_hospital,cnt_school,cnt_company
0,바나프레소 테헤란로점,커피숍,2019-05-23,서울 강남구 테헤란로 208,1.0,98.40,37.501130,127.039099,TR,3120197.0,11680650,banapresso,985131.50,882135.00,752222.8125,0,1,0,0
1,바나프레소 신논현역점,커피숍,2019-08-06,서울 서초구 사평대로 364,1.0,107.24,37.503733,127.023196,TR,3120187.0,11650531,banapresso,589152.00,788268.00,560948.1250,0,1,0,0
2,바나프레소 선릉공원점,커피숍,2021-07-23,서울 강남구 테헤란로63길 20,1.0,97.28,37.506522,127.050398,TR,3120210.0,11680590,banapresso,951358.25,983513.00,833761.1250,1,2,0,0
3,바나프레소 충무로점,커피숍,2022-09-19,서울 중구 필동로 15,1.0,148.43,37.560339,126.996257,TR,3120032.0,11140570,banapresso,357098.25,362848.50,287171.8750,0,0,0,0
4,바나프레소 홍대입구역사거리점,커피숍,2022-11-07,서울 마포구 양화로 129,1.0,33.00,37.555061,126.920694,TR,3120102.0,11440660,banapresso,409679.75,552486.25,389756.6250,0,1,0,0


# 3. 접근성 점수 변수
    3-1. 지하철역 최소 거리
    3-2. 버스정류장 최소 거리
    3-3. 대로변 여부
    3-4. 횡단보도 수

In [49]:
bus_stop = pd.read_csv("C:/vs_code_prj/brewmap/data/dh/bus_stop.csv")
station = pd.read_csv("C:/vs_code_prj/brewmap/data/dh/station.csv")
crosswalk = pd.read_csv("C:/vs_code_prj/brewmap/data/dh/crosswalk.csv")
road = pd.read_csv("C:/vs_code_prj/brewmap/data/dh/road.csv")

print(bus_stop.columns)
print(station.columns)
print(crosswalk.columns)
print(road.columns)

Index(['bus_stop_id', 'bus_stop_nm', 'lng', 'lat', 'type'], dtype='str')
Index(['addr', 'type', 'lat', 'lng'], dtype='str')
Index(['lng', 'lat', 'sigungu', 'emd', 'type'], dtype='str')
Index(['road_address', 'lng', 'lat', 'type'], dtype='str')


In [92]:
# 최소 거리
def min_distance(cafe_lat, cafe_lon, df):
    distances = df.apply(
        lambda row: haversine(
            (cafe_lat, cafe_lon), (row["lat"], row["lng"]), unit=Unit.METERS
        ), axis=1
    )
    return distances.min()

# 300m 안 횡단보도 개수
def count_within(cafe_lat, cafe_lon, df, radius=300):
    mask = df.apply(
        lambda row: haversine(
            (cafe_lat, cafe_lon), (row["lat"], row["lng"]), unit=Unit.METERS
        ) <= radius, axis=1
    )
    return mask.sum()

# 300m 안 대로변 여부
def is_roadside(cafe_lat, cafe_lon, df, radius=300):
    mask = df.apply(
        lambda row: haversine(
            (cafe_lat, cafe_lon), (row["lat"], row["lng"]), unit=Unit.METERS
        ) <= radius, axis=1
    )
    return int(mask.any())  # 1개라도 있으면 1, 없으면 0

# bana_old_df에 적용
bana_old_df["dist_station"]    = bana_old_df.apply(lambda row: min_distance(row["lat"], row["lng"], station), axis=1)
bana_old_df["dist_bus_stop"]   = bana_old_df.apply(lambda row: min_distance(row["lat"], row["lng"], bus_stop), axis=1)
bana_old_df["cnt_crosswalk"] = bana_old_df.apply(lambda row: count_within(row["lat"], row["lng"], crosswalk), axis=1)
bana_old_df["is_road"]     = bana_old_df.apply(lambda row: is_roadside(row["lat"], row["lng"], road), axis=1)
bana_old_df

,nm,type,apv_date,addr,floor,sitearea,lat,lng,flpop_type,trdar_cd,...,flpop_dinner,flpop_etc,cnt_academy,cnt_hospital,cnt_school,cnt_company,dist_station,dist_bus_stop,cnt_crosswalk,is_road
0,바나프레소 테헤란로점,커피숍,2019-05-23,서울 강남구 테헤란로 208,1.0,98.40,37.501130,127.039099,TR,3120197.0,...,882135.00,752222.8125,0,1,0,0,223.598018,24.766102,8,1
1,바나프레소 신논현역점,커피숍,2019-08-06,서울 서초구 사평대로 364,1.0,107.24,37.503733,127.023196,TR,3120187.0,...,788268.00,560948.1250,0,1,0,0,765.979395,57.031538,18,1
2,바나프레소 선릉공원점,커피숍,2021-07-23,서울 강남구 테헤란로63길 20,1.0,97.28,37.506522,127.050398,TR,3120210.0,...,983513.00,833761.1250,1,2,0,0,258.610257,139.653177,16,1
3,바나프레소 충무로점,커피숍,2022-09-19,서울 중구 필동로 15,1.0,148.43,37.560339,126.996257,TR,3120032.0,...,362848.50,287171.8750,0,0,0,0,209.265490,165.181903,21,1
4,바나프레소 홍대입구역사거리점,커피숍,2022-11-07,서울 마포구 양화로 129,1.0,33.00,37.555061,126.920694,TR,3120102.0,...,552486.25,389756.6250,0,1,0,0,332.299469,58.952811,36,1
5,바나프레소 양재이안점,커피숍,2023-01-31,서울 서초구 강남대로34길 7,1.0,48.33,37.483032,127.036465,TR,3120179.0,...,473092.50,400709.2500,0,1,0,0,196.929245,68.322979,12,1
6,바나프레소 여의도파크원점,커피숍,2021-03-18,서울 영등포구 여의대로 108,-1.0,59.16,37.525191,126.929113,TR,3120149.0,...,317197.25,246954.3125,0,0,0,0,392.144845,28.172926,27,1
7,바나프레소 양평역점,커피숍,2015-08-05,서울 영등포구 양산로 43,1.0,59.40,37.525761,126.887942,TR,3120135.0,...,68874.00,71096.8750,0,0,0,5,98.907836,50.168375,24,1
8,바나프레소 명동중앙우체국점,커피숍,2021-10-29,서울 중구 명동8나길 51,1.0,193.85,37.561080,126.982077,TR,3120028.0,...,410975.75,273562.8125,0,1,0,0,341.656695,57.127609,16,1
9,바나프레소 마포역점,커피숍,2015-11-17,서울 마포구 마포대로 34,1.0,33.00,37.539260,126.946152,TR,3120106.0,...,141110.00,142316.5000,0,1,0,0,41.330683,61.846039,31,1


# 4. 경쟁사 비교 점수 변수

In [56]:
cafe_nm = cafe['nm']
cafe_nm = cafe_nm.to_list()
cafe_nm

['베레베레',
 '덕수궁전통차전문',
 '엘빠소',
 '씨애틀즈베스트커피대사관점',
 '본솔카페',
 '링고요고',
 '토넷트커피숍',
 '썬휴게실',
 '네스카페',
 '루비콘플러스',
 '구스토',
 '아름다운고궁',
 '피카디리망고 일리',
 '소그노스에스프레소',
 '노모토커피',
 '주)키커피코리아',
 '강남커피숍',
 '캔모아',
 '북카페내서재',
 '플로리안',
 'BABIANA(바비아나)',
 '숭실대생협 아름다운세상',
 '보보스',
 '에스프레소인뉴욕',
 '할리스커피(Hollys Coffee)',
 '제일상회',
 '국립고궁박물관 뮤지엄숍',
 '모드뜨레',
 '카페돌로미티',
 '카페하임',
 '미래월드',
 '커피501(coffee501)',
 '메리메이트(MerryMate)',
 '귀천',
 '탐앤탐스노원점',
 '커피투어(Coffee Tour)',
 '쿠벅양평점',
 '세리커피(Coffee)',
 '테라 스위트(Terra sweet)본점',
 '물커피(Mool Coffee)',
 '세븐일레븐(서울시립대점)',
 '#11(Sharp eleven)',
 '커피보노',
 'ATM(에이티엠)',
 '미르엠(MIRM)',
 '토프레소 태평로점',
 'POLE243',
 'Meister haus',
 '빈스빈스',
 '이디야커피',
 '소풍',
 '다동커피집',
 '파나마',
 '동네커피',
 '영화에 빠진 커피',
 '두루',
 '니들 북',
 '헤이리커피공장103이촌점',
 '퓨얼리데카던트',
 '이디야올림픽공원점',
 '카페베네 종로5가점',
 '이디야 서대문점',
 '남산커피집',
 '보티',
 '커피데이',
 '아산엠 매점',
 '까사안띠구아',
 '커피홀릭',
 '카페깔리아리',
 '파반커피',
 '카페나루',
 '카페라비',
 '투미',
 '호텔디아망',
 '빈스커피',
 '투썸플레이스',
 '야노아',
 'An.s coffee',
 '슐레스',
 '알엔비스커피(RNBS COFFEE)',
 '아임빈',
 '바오밥나무 여성미래센

In [85]:
# 1) 이름 정규화 함수
def norm(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r"[\s\.\,\-\(\)\[\]\/_]", "", s)  # 공백/기호 제거
    return s

# 2) 브랜드 키워드(필요시 추가/수정)
LOW_PATTERNS = [
    "메가엠지씨", "메가mgc", "메가커피",
    "빽다방",
    "컴포즈", "compose",
    "더벤티", "theventi",
    "매머드", "mammoth",
    "텐퍼센트", "tenpercent",
    "커피베이",
    "우지커피", "oozy",
    "백억커피",
    "벤티프레소", "ventipresso",
]

HIGH_PATTERNS = [
    "스타벅스", "starbucks",
    "투썸", "twosome",
    "폴바셋", "paulbassett",
    "커피빈", "coffeebean",
    "할리스", "hollys",
    "아티제", "artisee",
    "테라로사", "terarosa",
    "오설록", "osulloc",
]

BANA_PATTERNS = ["바나프레소", "banapresso"]

def contains_any(text, patterns):
    return any(p in text for p in patterns)

# 3) 분류 함수
def cafe_tier(name):
    x = norm(name)
    if contains_any(x, BANA_PATTERNS):
        return "banapresso"
    if contains_any(x, HIGH_PATTERNS):
        return "high"
    if contains_any(x, LOW_PATTERNS):
        return "low"
    return "etc"

# 4) 적용
cafe["detail"] = cafe["nm"].apply(cafe_tier)

# 확인
print(cafe["detail"].value_counts())
cafe.head()

detail
etc           11197
low            2209
high           1071
banapresso       59
Name: count, dtype: int64


,nm,type,apv_date,addr,floor,sitearea,lat,lng,flpop_type,trdar_cd,adstrd_cd,detail
0,베레베레,커피숍,2002-09-19,서울 광진구 능동로13길 19,NaN,NaN,37.542885,127.070832,TR,3120053.0,11215710,etc
1,덕수궁전통차전문,커피숍,2003-01-16,서울 중구 서소문로 109,2.0,46.28,37.563062,126.973280,TR,3120020.0,11140520,etc
2,엘빠소,커피숍,2003-05-31,서울 종로구 성균관로 18,NaN,NaN,37.584406,126.997641,TR,3110021.0,11110650,etc
3,씨애틀즈베스트커피대사관점,커피숍,2003-10-31,서울 종로구 세종대로 188,1.0,NaN,37.573190,126.977831,AD,NaN,11110615,etc
4,본솔카페,커피숍,2004-01-17,서울 용산구 청파로47길 71,1.0,13.80,37.544840,126.966455,TR,3110069.0,11170555,etc


In [93]:
# 0) detail 매핑
detail_map = {
    "banapresso": "cnt_banapresso",
    "high": "cnt_high_cafe",
    "low": "cnt_low_cafe",
    "etc": "cnt_etc_cafe"
}

# 1) 함수 정의
def count_cafes_nearby(cafe_lat, cafe_lon, cafe_idx, cafes_df, radius=300):
    # 자기 자신 제외
    other_cafes = cafes_df.drop(index=cafe_idx)
    
    mask = other_cafes.apply(
        lambda row: haversine(
            (cafe_lat, cafe_lon), (row["lat"], row["lng"]), unit=Unit.METERS
        ) <= radius, axis=1
    )
    nearby = other_cafes[mask]
    counts = nearby.groupby("detail").size()
    
    # 없는 type도 0으로 채우기
    counts = counts.reindex(detail_map.keys(), fill_value=0)
    counts.index = [detail_map[d] for d in counts.index]
    return counts

# 2) 적용
result = bana_old_df.apply(
    lambda row: count_cafes_nearby(row["lat"], row["lng"], row.name, cafe),
    axis=1
).fillna(0).astype(int)

bana_old_df = pd.concat([bana_old_df, result], axis=1)
bana_old_df

,nm,type,apv_date,addr,floor,sitearea,lat,lng,flpop_type,trdar_cd,...,cnt_school,cnt_company,dist_station,dist_bus_stop,cnt_crosswalk,is_road,cnt_banapresso,cnt_high_cafe,cnt_low_cafe,cnt_etc_cafe
0,바나프레소 테헤란로점,커피숍,2019-05-23,서울 강남구 테헤란로 208,1.0,98.40,37.501130,127.039099,TR,3120197.0,...,0,0,223.598018,24.766102,8,1,1,6,6,27
1,바나프레소 신논현역점,커피숍,2019-08-06,서울 서초구 사평대로 364,1.0,107.24,37.503733,127.023196,TR,3120187.0,...,0,0,765.979395,57.031538,18,1,1,6,8,36
2,바나프레소 선릉공원점,커피숍,2021-07-23,서울 강남구 테헤란로63길 20,1.0,97.28,37.506522,127.050398,TR,3120210.0,...,0,0,258.610257,139.653177,16,1,1,1,10,25
3,바나프레소 충무로점,커피숍,2022-09-19,서울 중구 필동로 15,1.0,148.43,37.560339,126.996257,TR,3120032.0,...,0,0,209.265490,165.181903,21,1,1,4,5,31
4,바나프레소 홍대입구역사거리점,커피숍,2022-11-07,서울 마포구 양화로 129,1.0,33.00,37.555061,126.920694,TR,3120102.0,...,0,0,332.299469,58.952811,36,1,1,4,4,28
5,바나프레소 양재이안점,커피숍,2023-01-31,서울 서초구 강남대로34길 7,1.0,48.33,37.483032,127.036465,TR,3120179.0,...,0,0,196.929245,68.322979,12,1,1,6,5,20
6,바나프레소 여의도파크원점,커피숍,2021-03-18,서울 영등포구 여의대로 108,-1.0,59.16,37.525191,126.929113,TR,3120149.0,...,0,0,392.144845,28.172926,27,1,1,3,1,24
7,바나프레소 양평역점,커피숍,2015-08-05,서울 영등포구 양산로 43,1.0,59.40,37.525761,126.887942,TR,3120135.0,...,0,5,98.907836,50.168375,24,1,1,1,2,15
8,바나프레소 명동중앙우체국점,커피숍,2021-10-29,서울 중구 명동8나길 51,1.0,193.85,37.561080,126.982077,TR,3120028.0,...,0,0,341.656695,57.127609,16,1,1,5,4,62
9,바나프레소 마포역점,커피숍,2015-11-17,서울 마포구 마포대로 34,1.0,33.00,37.539260,126.946152,TR,3120106.0,...,0,0,41.330683,61.846039,31,1,1,2,4,14


# 중간 저장(csv로 내보내기)

In [94]:
bana_old_df.to_csv("bana_old.csv", index=False, encoding="utf-8-sig")

# 5. 수요 + 접근성 + 경쟁사 점수